# 01. 取り込みと素性調査

みなとストア(3店舗)の4月の売上を、**3つの系統から受け取って1枚にまとめる**のがこの章です。

## この章のゴール

```
  旧POS  sales_2024-04_old.csv   14行  ┐
  新POS  sales_2024-04_new.csv   12行  ├→  1枚の表 36行 (全部文字列)
  EC     sales_2024-04_ec.csv    10行  ┘    + どのファイル由来かの列
```

出口はこの8列です。この形は次の章以降でもずっと使います。

| 列 | 中身 |
| --- | --- |
| `sale_date` | 売上日(この時点では書式バラバラのまま) |
| `shop_name` | 店名(表記ゆれたまま) |
| `item_cd` | 商品コード |
| `qty` | 数量 |
| `amount` | 金額 |
| `tax_type` | 税込か税抜か |
| `note` | 備考 |
| `source` | どのファイルから来たか |

## この章の進め方

**考え方を読む → セルを実行する → 出力の読み方を知る → 1問だけ書いてみる**、の繰り返しです。

「書いてみる」は直前にやったことを別の列でなぞるだけの軽さにしてあります。
分からなければ、すぐ下の**答え**を開いて写してかまいません。それで十分身に付きます。

> この章では**まだ何も直しません。** 汚れは汚れのまま、1枚に集めるところまでです。
> 直すのは次の章です。

In [ ]:
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

---
## 1. まず現物を見る

pandas に読ませる前に、**テキストとしてそのまま見ます。**

いきなり `read_csv` すると、pandas が気を利かせて型を変えたり欠損を埋めたりするので、
「元は何だったのか」が見えなくなります。事故の大半はここで防げます。

### 1-1. 新POSのファイルを開く

In [ ]:
print(open("/data/sales_2024-04_new.csv", encoding="utf-8").read())

出力を見てください。この時点で気づいてほしいのは3つです。

- 金額が `900` のものと `"￥1,200"` のものがある。**同じ列なのに書き方が違う**
- 数量に `-`、金額に `N/A` がある。**欠損の表し方が2種類ある**
- 日付が `2024/4/1`。ゼロ埋めされていない

まだ直しません。**「そういうものが入っている」と分かれば、いまはそれで十分です。**

In [ ]:
# ✍ 書いてみる: EC のファイル (sales_2024-04_ec.csv) を同じように開いて、
#              中身を目で見てください。

ans = ...   # ここに書く

assert "EC-0001" in ans
assert "返品" in ans, "返品の行があるはずです"
print("OK")

<details>
<summary>答え</summary>

```python
ans = open("/data/sales_2024-04_ec.csv", encoding="utf-8").read()
print(ans)
```

</details>

EC のファイルには**返品の行**があります。数量も金額もマイナスです。
これは間違いではなく、そういう仕様です。4章の集計で効いてきます。

### 1-2. 旧POSは、そのままでは開けない

同じように旧POSを開いてみます。**エラーになります。**

In [ ]:
try:
    print(open("/data/sales_2024-04_old.csv", encoding="utf-8").read())
except Exception as e:
    print(f"{type(e).__name__}: {e}")

`UnicodeDecodeError` です。

日本語のファイルは **UTF-8 とは限りません。** 古いシステムから出てくるものは
**cp932(Shift_JIS)** であることがよくあります。読む側が指定してあげる必要があります。

> エラーメッセージの `invalid start byte` は、
> 「UTF-8 として読もうとしたら、UTF-8 ではありえないバイトが出てきた」という意味です。
> **文字コードが違うときの、いちばん典型的なエラー**なので、見た瞬間に疑えるようにしておきます。

In [ ]:
print(open("/data/sales_2024-04_old.csv", encoding="cp932").read())

読めました。そして中身がまた違います。

- 商品CD・数量・金額が**全角数字**(`０００１` `２` `９００`)
- 日付が `2024年4月1日`
- 店名が `ミナトストア　渋谷`。**空白が全角**で、しかも新POSの `みなとストア渋谷店` と別表記

同じ会社の同じ店なのに、系統ごとに表記が違います。
**送信元は直してくれません。** 受け取る側で吸収します。

In [ ]:
# ✍ 書いてみる: 旧POSを cp932 で開いて、先頭2行だけ表示してください。
#              (ヒント: 読んだ文字列を .splitlines() で行のリストにできます)

ans = ...   # ここに書く

assert len(ans) == 2, f"2行のはずです: {len(ans)}"
assert ans[0].startswith("売上日")
print("OK")
print(ans)

<details>
<summary>答え</summary>

```python
ans = open("/data/sales_2024-04_old.csv", encoding="cp932").read().splitlines()[:2]
```

</details>

---
## 2. pandas で読む

現物を見たので、ここから pandas に載せます。

**読み込みの設定は、最初はこの2つだけ覚えれば大丈夫です。**

| 設定 | 何のため |
| --- | --- |
| `dtype=str` | pandas に型を推測させない。**推測は汚れを勝手に埋めてしまう** |
| `keep_default_na=False` | `N/A` などを勝手に欠損にしない。欠損の判定は自分でやる |

なぜそうするかというと、**この段階では「元のまま」を保ちたい**からです。
勝手に数値にされてしまうと、`00`5 の先頭ゼロが消えたり、
`N/A` が欠損に変わったりして、**元が何だったのか分からなくなります**。

### 2-1. 1ファイル読んでみる

In [ ]:
new = pd.read_csv("/data/sales_2024-04_new.csv", dtype=str, keep_default_na=False)
print(new.shape)
new

`(12, 6)` — 12行6列です。表示された値が、さっきテキストで見たものと**同じ**なのを確認してください。
`￥1,200` も `N/A` も `-` も、そのまま文字列として入っています。これが狙いどおりの状態です。

In [ ]:
# ✍ 書いてみる: EC のファイルを、同じ設定 (dtype=str, keep_default_na=False) で読んでください。

ans = ...   # ここに書く

assert ans.shape == (10, 7), f"(10, 7) のはずです: {ans.shape}"
assert ans["金額税抜"].iloc[8] == "-500", "返品の金額がマイナスのまま残っているはずです"
print("OK")

<details>
<summary>答え</summary>

```python
ans = pd.read_csv("/data/sales_2024-04_ec.csv", dtype=str, keep_default_na=False)
```

</details>

### 2-2. 3ファイルの列名を並べて比べる

3つとも「売上のファイル」ですが、**列の名前も順番も違います。**
これは実務ではふつうのことです。まず並べて見ます。

In [ ]:
old = pd.read_csv("/data/sales_2024-04_old.csv", dtype=str, keep_default_na=False,
                  encoding="cp932")
new = pd.read_csv("/data/sales_2024-04_new.csv", dtype=str, keep_default_na=False)
ec  = pd.read_csv("/data/sales_2024-04_ec.csv",  dtype=str, keep_default_na=False)

for name, df in [("旧POS", old), ("新POS", new), ("EC", ec)]:
    print(f"{name:5} {len(df):3}行  {df.columns.tolist()}")

見比べると:

- **旧POSと新POSは、列名は同じでも順番が違います**(`売上日` が先か後か)
- **EC だけ列名が違います**(`受注日` `金額税抜` `ステータス`)、しかも `受注番号` という余分な列がある
- EC には `備考` がありません

> ここで**「列の順番」で処理を書かないこと**が大事です。
> `df.iloc[:, 0]` のように位置で取ると、ファイルが1つ増えただけで壊れます。
> **必ず列名で取ります。**

In [ ]:
# ✍ 書いてみる: 旧POS (old) の列名をリストで取り出してください。

ans = ...   # ここに書く

assert ans == ["売上日", "店舗名", "商品CD", "数量", "金額", "備考"], ans
print("OK")

<details>
<summary>答え</summary>

```python
ans = old.columns.tolist()
```

</details>

---
## 3. 列を揃えて1枚にする

ここからが取り込みの本体です。**3つの表を同じ形にしてから、縦に積みます。**

考え方はこうです。

```
バラバラの列名   →   共通の列名に付け替える   →   縦に積む
   (rename)                                      (concat)
```

共通の列名(この章のゴールの8列)を決めて、そこへ寄せます。
**決めるのは自分**です。あとの章がずっと使うので、ここで決めた名前が効いてきます。

### 3-1. 列名を付け替える

`rename(columns={...})` に「元の名前 → 新しい名前」の辞書を渡します。

In [ ]:
old2 = old.rename(columns={
    "売上日": "sale_date",
    "店舗名": "shop_name",
    "商品CD": "item_cd",
    "数量": "qty",
    "金額": "amount",
    "備考": "note",
})
old2.head(3)

列名が英語になりました。中身は何も変わっていません。

**なぜ英語にするのか:** このあと `df.qty` のように書けること、
Parquet に書き出すときに扱いやすいこと、の2つが理由です。
日本語のままでも動きますが、慣習として英語にしておくと後が楽です。

In [ ]:
# ✍ 書いてみる: EC (ec) の列名を、共通の名前に付け替えてください。
#              受注日→sale_date, 店舗名→shop_name, 商品CD→item_cd,
#              数量→qty, 金額税抜→amount, ステータス→note

ans = ...   # ここに書く

assert "sale_date" in ans.columns and "amount" in ans.columns
assert ans["note"].iloc[8] == "返品"
print("OK")

<details>
<summary>答え</summary>

```python
ans = ec.rename(columns={
    "受注日": "sale_date",
    "店舗名": "shop_name",
    "商品CD": "item_cd",
    "数量": "qty",
    "金額税抜": "amount",
    "ステータス": "note",
})
```

</details>

### 3-2. 足りない情報を列として足す

3つの表には、**表に書かれていない大事な違い**があります。

- 店舗のPOS(旧・新)の金額は**税込**
- EC の金額は**税抜**

これは列になっていないので、このままでは合計したときに黙って狂います。
**分かっているうちに、列として持たせておきます。**

同じ理由で「どのファイルから来たか」も列にします。
あとで数が合わないとき、**どの系統が原因かを追える**ようにするためです。

In [ ]:
COLUMNS = ["sale_date", "shop_name", "item_cd", "qty", "amount",
           "tax_type", "note", "source"]

old2 = old2.assign(tax_type="税込", source="old")[COLUMNS]
old2.head(3)

`assign` で2列足して、`[COLUMNS]` で**列の順番も揃えて**います。

> `[COLUMNS]` のように列名のリストで取り出すと、**そのリストの順に並びます**。
> 3つの表で同じリストを使えば、列の順番が自動的に揃います。

In [ ]:
# ✍ 書いてみる: 新POS (new) を、共通の8列の形にしてください。
#              列名を付け替え、tax_type="税込"、source="new" を足し、COLUMNS の順に並べます。

ans = ...   # ここに書く

assert ans.columns.tolist() == COLUMNS, ans.columns.tolist()
assert len(ans) == 12
assert (ans["source"] == "new").all()
print("OK")

<details>
<summary>答え</summary>

```python
ans = new.rename(columns={
    "売上日": "sale_date",
    "店舗名": "shop_name",
    "商品CD": "item_cd",
    "数量": "qty",
    "金額": "amount",
    "備考": "note",
}).assign(tax_type="税込", source="new")[COLUMNS]
```

</details>

### 3-3. 縦に積む

3つとも同じ8列になったら、`pd.concat` でつなぎます。

In [ ]:
# 3つを同じ形にする (上でやったことをまとめて書いています)
new2 = new.rename(columns={
    "売上日": "sale_date", "店舗名": "shop_name", "商品CD": "item_cd",
    "数量": "qty", "金額": "amount", "備考": "note",
}).assign(tax_type="税込", source="new")[COLUMNS]

ec2 = ec.rename(columns={
    "受注日": "sale_date", "店舗名": "shop_name", "商品CD": "item_cd",
    "数量": "qty", "金額税抜": "amount", "ステータス": "note",
}).assign(tax_type="税抜", source="ec")[COLUMNS]

raw = pd.concat([old2, new2, ec2], ignore_index=True)
print(raw.shape)
raw

36行になりました。**14 + 12 + 10 = 36** です。1行も落ちていません。

> `ignore_index=True` を付けないと、index が `0,1,2,...,13,0,1,2,...` と重複します。
> 積んだあとは振り直しておくのが無難です。

**必ず行数を確かめてください。** 「読めた」と「全部読めた」は違います。
足し算が合わなければ、どこかで落ちています。

In [ ]:
# ✍ 書いてみる: raw を source ごとに数えて、内訳を出してください。

ans = ...   # ここに書く

assert ans.to_dict() == {"old": 14, "new": 12, "ec": 10}, ans.to_dict()
print("OK")

<details>
<summary>答え</summary>

```python
ans = raw["source"].value_counts()
```

</details>

---
## 4. 素性を調べる

1枚になりました。ここで**すぐ次に進まない**のが、この章でいちばん大事なところです。

**読めた ≠ 正しい。** 何が入っているかを先に数えます。
これを**素性調査(プロファイリング)**と呼びます。仕様書を読んで想像するより、
データに聞いたほうが速くて確実です。

調べる順番はだいたい決まっています。

| 順番 | 見るもの | 何が分かるか |
| --- | --- | --- |
| 1 | 大きさと型 | 想定どおりの行数か |
| 2 | 欠損 | どの列がどれくらい空か |
| 3 | 値の種類 | 表記ゆれ、想定外の値 |
| 4 | 数値にできるか | あとで計算できない値がどれか |
| 5 | キーの重複 | 1行が何を表しているか |

### 4-1. 大きさと型

In [ ]:
print("行数と列数:", raw.shape)
print()
print(raw.dtypes)

全部 `object`(文字列)です。**そうなるように読んだ**ので、これで正しい状態です。

数値になっていないので、いまは `raw["amount"].sum()` のような計算はできません。
それは次の章でやります。

### 4-2. 欠損を数える

ここで注意があります。`keep_default_na=False` で読んだので、
**空欄は「欠損」ではなく「空文字」**として入っています。

In [ ]:
print("isna() で数えた欠損:")
print(raw.isna().sum())
print()
print("空文字の数:")
print((raw == "").sum())

`isna()` は**全部ゼロ**です。空文字は欠損ではないからです。

これは「欠損が無い」という意味ではありません。**まだ欠損として扱っていない**だけです。
`note` の空文字は「備考なし」で正常ですが、`qty` の `-` や `amount` の `N/A` は明らかに欠損です。

**何を欠損と見なすかは、データを見てから自分で決めます。** 次の章の最初にやります。

In [ ]:
# ✍ 書いてみる: amount 列に "N/A" がいくつあるか数えてください。

ans = ...   # ここに書く

assert ans == 1, f"1件のはずです: {ans}"
print("OK")

<details>
<summary>答え</summary>

```python
ans = (raw["amount"] == "N/A").sum()
```

</details>

### 4-3. 値の種類を見る

`value_counts()` は、**表記ゆれを見つけるための道具**です。

In [ ]:
print(raw["shop_name"].value_counts())

店は3店舗のはずなのに、**8種類**出てきました。

```
みなとストア渋谷店        ← 新POS
ミナトストア　渋谷        ← 旧POS (カタカナ + 全角空白)
みなと渋谷               ← EC (略称)
(株)みなとストア 新宿店   ← 新POS (会社名つき)
みなとストア大宮店        ← 閉店したはずの店!
```

**同じ店が別の名前で入っています。** このまま店舗ごとに集計すると、
渋谷店の売上が3つに割れます。これを1つに寄せるのが**名寄せ**で、3章でやります。

大宮店の行も見つかりました。閉店した店の売上が立っています。これも3章で扱います。

> `value_counts()` を最初に叩く癖をつけてください。
> **表記ゆれは、数えれば必ず見つかります。**

In [ ]:
# ✍ 書いてみる: item_cd の値の種類と件数を出してください。

ans = ...   # ここに書く

assert ans.sum() == 36
assert len(ans) >= 10, "全角のコードと半角のコードが別々に数えられるはずです"
print("OK")
print(ans)

<details>
<summary>答え</summary>

```python
ans = raw["item_cd"].value_counts()
```

</details>

商品コードも揺れています。

- `０００１`(旧POSの全角)
- `0001`(新POSの半角)
- `1`(ECは**先頭ゼロが落ちている**)

**3つとも同じ商品**です。これも次の章で揃えます。

### 4-4. 数値にできない値を洗い出す

金額はあとで必ず合計します。**いま数値にできない値がどれかを、先に知っておきます。**

`pd.to_numeric(..., errors="coerce")` は、数値にできない値を `NaN` にします。
その `NaN` になった**元の値**を見れば、何が邪魔しているか分かります。

In [ ]:
num = pd.to_numeric(raw["amount"], errors="coerce")
bad = raw.loc[num.isna(), "amount"]

print("数値にできない値の数:", len(bad))
print()
print(bad.value_counts())

出てきたのは3種類です。

| 値 | 理由 |
| --- | --- |
| `９００` などの全角 | pandas は全角数字を数値と見なしません |
| `￥1,200` | 通貨記号とカンマが混ざっている |
| `N/A` | そもそも数値ではない(欠損) |

**この3つを潰せば金額は数値になる**、と分かりました。これが素性調査の成果です。
次の章では、この一覧を見ながら直していきます。

In [ ]:
# ✍ 書いてみる: 同じやり方で、qty (数量) のうち数値にできない値を出してください。

ans = ...   # ここに書く

assert set(ans.unique()) <= {"２", "１", "３", "４", "-"}, ans.unique().tolist()
assert "-" in ans.tolist(), "欠損を表す - が含まれるはずです"
print("OK")
print(ans.value_counts())

<details>
<summary>答え</summary>

```python
num_qty = pd.to_numeric(raw["qty"], errors="coerce")
ans = raw.loc[num_qty.isna(), "qty"]
```

</details>

### 4-5. 日付の書式が何種類あるか

日付も同じで、**書式が混ざったままでは変換できません。**
どんな書式が来ているのかを見ます。

In [ ]:
# source ごとに、日付の最初の1件を見る
print(raw.groupby("source")["sale_date"].first())

3系統で3書式です。

| source | 例 | 書式 |
| --- | --- | --- |
| `old` | `2024年4月1日` | 和風 |
| `new` | `2024/4/1` | スラッシュ、ゼロ埋めなし |
| `ec` | `2024-04-02` | ISO |

**1つの `to_datetime` では変換できません。** 書式ごとに変換して重ねる、
というやり方を次の章でやります。

### 4-6. 1行が何を表しているか(粒度)

最後に、**この表の1行は何なのか**を確かめます。ここが曖昧なまま集計すると、
二重計上や取りこぼしが起きます。

この表は「1行 = 1回の売上明細」のはずです。だとすると、
同じ日・同じ店・同じ商品の行が2つあっても**おかしくありません**(その日に2回売れた)。
本当にそうなっているか、重複を数えてみます。

In [ ]:
key = ["sale_date", "shop_name", "item_cd"]
dup = raw[raw.duplicated(key, keep=False)]

print("重複している行数:", len(dup))

**0件**でした。

ここで「重複は無い」と結論づけると間違いです。**そう見えているだけ**です。

思い出してください。店名は `ミナトストア　渋谷` と `みなとストア渋谷店` と `みなと渋谷` に割れていて、
商品コードは `０００１` と `0001` と `1` に割れていて、日付は `2024年4月1日` と `2024/4/1` に割れています。

**同じものが同じだと判定されていないので、重複のしようがない**のです。

> **重複チェックは、表記を整えたあとでなければ意味を持ちません。**
> 順番が大事、というのはこういうことです。整える(02章)→ 名寄せする(03章)→
> そのあとでもう一度この確認に戻ってきます。

いまの時点で言えるのは、「**この表の粒度は明細である**」ということだけです。
それでも、それが分かっていれば集計は組み立てられます。

In [ ]:
# ✍ 書いてみる: 「日付 × 店名」の組み合わせが何通りあるか数えてください。
#              (ヒント: 2列だけ取り出して drop_duplicates した行数)

ans = ...   # ここに書く

assert ans == 33, f"33通りのはずです: {ans}"
print("OK")

<details>
<summary>答え</summary>

```python
ans = len(raw[["sale_date", "shop_name"]].drop_duplicates())
```

</details>

**33通り**でした。

店は3つ、日付は4/1〜4/10 の10日ぶん。多くても 3 × 10 = **30通り**のはずです。
それを超えているのは、**店名も日付も表記が割れて、別物として数えられているから**です。

こういう「**ありえない数字**」に気づけるかどうかが、素性調査の勘所です。
数えた結果を、自分の知っている事実(3店舗・10日)と突き合わせてください。

---
## 5. この章のまとめ

やったことを**関数1つ**にまとめます。ここまでのセルを上から順に読み直しながら、
「読む → 列名を揃える → 積む」の3つだけが入っていることを確認してください。

In [ ]:
COLUMNS = ["sale_date", "shop_name", "item_cd", "qty", "amount",
           "tax_type", "note", "source"]

# ファイルごとの違いを、辞書として1箇所にまとめておく
SOURCES = [
    {
        "path": "/data/sales_2024-04_old.csv",
        "encoding": "cp932",
        "tax_type": "税込",
        "source": "old",
        "rename": {"売上日": "sale_date", "店舗名": "shop_name", "商品CD": "item_cd",
                   "数量": "qty", "金額": "amount", "備考": "note"},
    },
    {
        "path": "/data/sales_2024-04_new.csv",
        "encoding": "utf-8",
        "tax_type": "税込",
        "source": "new",
        "rename": {"売上日": "sale_date", "店舗名": "shop_name", "商品CD": "item_cd",
                   "数量": "qty", "金額": "amount", "備考": "note"},
    },
    {
        "path": "/data/sales_2024-04_ec.csv",
        "encoding": "utf-8",
        "tax_type": "税抜",
        "source": "ec",
        "rename": {"受注日": "sale_date", "店舗名": "shop_name", "商品CD": "item_cd",
                   "数量": "qty", "金額税抜": "amount", "ステータス": "note"},
    },
]


# 1ファイルを読んで、共通の8列の形にして返す
def read_one(spec):
    df = pd.read_csv(spec["path"], dtype=str, keep_default_na=False,
                     encoding=spec["encoding"])
    df = df.rename(columns=spec["rename"])
    df = df.assign(tax_type=spec["tax_type"], source=spec["source"])
    return df[COLUMNS]


# 全ファイルを読んで縦に積む
def load_raw(sources=SOURCES):
    frames = [read_one(s) for s in sources]
    df = pd.concat(frames, ignore_index=True)
    print(f"取り込み: {len(df)}行  内訳 {df['source'].value_counts().to_dict()}")
    return df


raw = load_raw()
raw.head()

**辞書にまとめたのがポイントです。** ファイルが1つ増えても、
`SOURCES` に4つ目を足すだけで済みます。関数は書き換えません。

これが「取り込み層」の基本の形です。
**系統ごとの違いを設定として外に出し、処理は1本にする。**

In [ ]:
# ✍ 書いてみる: load_raw() の結果が、この章のゴールどおりか確かめてください。
#              行数36、列は COLUMNS のとおり、全部文字列。

ans = ...   # ここに書く

assert ans is True
print("OK")

<details>
<summary>答え</summary>

```python
ans = bool(
    len(raw) == 36
    and raw.columns.tolist() == COLUMNS
    and (raw.dtypes == object).all()
)
```

</details>

---
## この章で分かったこと

| | |
| --- | --- |
| 文字コード | 日本語のファイルは UTF-8 とは限らない。`UnicodeDecodeError` が出たら cp932 を疑う |
| 読み方 | 取り込み層では `dtype=str, keep_default_na=False`。**推測させない** |
| 列 | 位置ではなく**名前**で扱う。共通の列名に寄せてから積む |
| 積んだあと | **必ず行数を確かめる**(14 + 12 + 10 = 36) |
| 素性調査 | 大きさ → 欠損 → 値の種類 → 数値にできるか → キー、の順に見る |
| 粒度 | この表の1行は「1明細」。一意に指す列は無い |

## 次の章に持ち越す宿題

素性調査で見つかった、**直すべきもの**の一覧です。02章でこれを順に潰します。

- [ ] 欠損の表し方が3種類(空文字 / `-` / `N/A`)
- [ ] 全角数字(`０００１` `２` `９００`)
- [ ] 店名の全角空白と表記ゆれ
- [ ] 金額の `￥` とカンマ
- [ ] **税込と税抜が混ざっている**
- [ ] 日付が3書式
- [ ] 商品コードの先頭ゼロが落ちている系統がある

次: `02-clean.ipynb`